# TF-IDF vs Word2Vec vs FastText vs GloVe

**Görev:** Metin sınıflandırma (20 Newsgroups, 4 kategori)

Her yöntem metni bir vektöre çevirir, ardından aynı sınıflandırıcı (LogisticRegression) ile test edilir.

In [ ]:
# !pip install scikit-learn gensim numpy pandas matplotlib

In [ ]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from gensim.models import Word2Vec, FastText
import gensim.downloader as api

## 1) Veri

In [ ]:
kategoriler = ["sci.space", "rec.autos", "comp.graphics", "talk.politics.misc"]
data = fetch_20newsgroups(subset="all", categories=kategoriler,
                          remove=("headers", "footers", "quotes"))

X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42, stratify=data.target)

def tokenize(metin):
    return re.findall(r"[a-z]+", metin.lower())

print(f"Eğitim: {len(X_train)} doküman | Test: {len(X_test)} doküman")

## 2) Vektörleştiriciler

Hepsi aynı arayüze sahip: `fit(metinler)` / `transform(metinler)`.

### TF-IDF
Seyrek, yüksek boyutlu vektör. Kelime anlamı yok, sadece frekans ağırlığı.

In [ ]:
class TfidfYontemi:
    isim = "TF-IDF"

    def __init__(self):
        self.vec = TfidfVectorizer(max_features=20000, stop_words="english")

    def fit(self, metinler):
        self.vec.fit(metinler)
        return self

    def transform(self, metinler):
        return self.vec.transform(metinler)

### Ortak embedding sınıfı
Kelime vektörlerinin ortalaması alınarak doküman vektörü üretilir.

In [ ]:
class OrtalamaEmbedding:
    isim = "Embedding"

    def __init__(self):
        self.kv = None  # gensim KeyedVectors

    def _dokuman_vektoru(self, tokenlar):
        vektorler = [self.kv[t] for t in tokenlar if t in self.kv]
        if not vektorler:
            return np.zeros(self.kv.vector_size)
        return np.mean(vektorler, axis=0)

    def transform(self, metinler):
        return np.vstack([self._dokuman_vektoru(tokenize(m)) for m in metinler])

### Word2Vec
Veri üzerinde eğitilir (skip-gram).

In [ ]:
class Word2VecYontemi(OrtalamaEmbedding):
    isim = "Word2Vec"

    def fit(self, metinler):
        cumleler = [tokenize(m) for m in metinler]
        model = Word2Vec(cumleler, vector_size=100, window=5, min_count=2,
                         sg=1, epochs=10, workers=4, seed=42)
        self.kv = model.wv
        return self

### FastText
Alt-kelime (karakter n-gram) kullanır; sözlükte olmayan kelimelere de vektör üretebilir.

In [ ]:
class FastTextYontemi(OrtalamaEmbedding):
    isim = "FastText"

    def fit(self, metinler):
        cumleler = [tokenize(m) for m in metinler]
        model = FastText(cumleler, vector_size=100, window=5, min_count=2,
                         sg=1, epochs=10, workers=4, seed=42)
        self.kv = model.wv
        return self

    def _dokuman_vektoru(self, tokenlar):
        if not tokenlar:
            return np.zeros(self.kv.vector_size)
        return np.mean([self.kv[t] for t in tokenlar], axis=0)

### GloVe
Eğitilmez; önceden eğitilmiş model indirilir (ilk çalıştırmada ~130 MB).

In [ ]:
class GloVeYontemi(OrtalamaEmbedding):
    isim = "GloVe"

    def __init__(self, model_adi="glove-wiki-gigaword-100"):
        super().__init__()
        self.model_adi = model_adi

    def fit(self, metinler):
        self.kv = api.load(self.model_adi)
        return self

## 3) Değerlendirme

In [ ]:
def degerlendir(yontem, X_train, X_test, y_train, y_test):
    t0 = time.time()
    yontem.fit(X_train)
    Xtr = yontem.transform(X_train)
    Xte = yontem.transform(X_test)
    sure = time.time() - t0

    clf = LogisticRegression(max_iter=2000)
    clf.fit(Xtr, y_train)
    tahmin = clf.predict(Xte)

    return {
        "Yöntem": yontem.isim,
        "Accuracy": accuracy_score(y_test, tahmin),
        "F1 (macro)": f1_score(y_test, tahmin, average="macro"),
        "Boyut": Xtr.shape[1],
        "Süre (sn)": round(sure, 2),
    }

In [ ]:
yontemler = [TfidfYontemi(), Word2VecYontemi(), FastTextYontemi(), GloVeYontemi()]
sonuclar = []

for y in yontemler:
    print(f"Çalışıyor: {y.isim} ...")
    try:
        sonuclar.append(degerlendir(y, X_train, X_test, y_train, y_test))
    except Exception as e:
        print(f"  {y.isim} atlandı: {e}")

df = pd.DataFrame(sonuclar).sort_values("F1 (macro)", ascending=False).reset_index(drop=True)
df

## 4) Grafik

In [ ]:
ax = df.plot(x="Yöntem", y=["Accuracy", "F1 (macro)"], kind="bar", figsize=(8, 4), rot=0)
ax.set_ylim(0, 1)
ax.set_title("Yöntem karşılaştırması")
plt.tight_layout()
plt.show()

## 5) Anlamsal benzerlik
TF-IDF'te kelime vektörü olmadığı için sadece embedding yöntemleri listelenir.

In [ ]:
def benzer_kelimeler(yontemler, kelime, n=5):
    print(f"'{kelime}' kelimesine en yakın {n} kelime:")
    for y in yontemler:
        if getattr(y, "kv", None) is None:
            continue
        try:
            yakinlar = [w for w, _ in y.kv.most_similar(kelime, topn=n)]
            print(f"  {y.isim:<10}: {', '.join(yakinlar)}")
        except KeyError:
            print(f"  {y.isim:<10}: (kelime sözlükte yok)")
    print()

for k in ["car", "space", "graphics"]:
    benzer_kelimeler(yontemler, k)